<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

Install the third-party libraries we rely on so this notebook can pull market data, run a regression, and plot results end-to-end.

In [ ]:
!pip install yfinance statsmodels matplotlib numpy

If you are running this outside a notebook (or in a locked-down environment), you may need to install packages in your project environment instead of using pip magic. If pip fails due to SSL or compiler issues, use your platform’s package manager or a prebuilt environment like Anaconda.

## Imports and setup

We use numpy for numeric helpers, matplotlib for plotting, statsmodels for OLS regression (alpha/beta), yfinance to download TSLA and SPY prices, and statsmodels.regression for the specific OLS model class.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf
from statsmodels import regression

Pull daily price history for the asset (TSLA) and the market proxy (SPY) so we can measure how much TSLA moves with the market.

In [ ]:
data = yf.download(["TSLA", "SPY"], start="2014-01-01", end="2015-01-01")
closes = data.Close
asset = closes.TSLA
benchmark = closes.SPY

Using SPY as the benchmark makes “market background noise” concrete, which is the whole point of beta: quantify the shared risk. In real research, you’d be deliberate about the benchmark choice because beta is always “relative to something.”

Convert prices into daily returns, which is the right scale for a linear beta model and makes the regression interpretable as “percent move vs percent move.”

In [ ]:
asset_returns = asset.pct_change().dropna()
benchmark_returns = benchmark.pct_change().dropna()
X = benchmark_returns.values
Y = asset_returns.values

Beginners often regress prices by mistake and get misleading relationships driven by trends. Returns also keep the hedge logic consistent, because hedging is about offsetting day-to-day PnL swings, not matching price levels.

## Estimate alpha and beta regression

Wrap the OLS fit into a small helper so we can reuse the exact same alpha/beta estimation on both the raw asset and the hedged portfolio.

In [ ]:
def linreg(x, y):
    x = sm.add_constant(x)
    model = regression.linear_model.OLS(y, x).fit()
    x = x[:, 1]
    return model.params[0], model.params[1]

This regression is the “separate market vs stock-specific” step: beta is the slope on SPY, and alpha is what’s left after accounting for SPY’s move. Even if we never trade the hedge, this is how we check whether our strategy is real signal or just disguised market exposure.

Fit the model on TSLA vs SPY so we can see how much of TSLA’s daily movement is explained by the market.

In [ ]:
alpha, beta = linreg(X, Y)
print(f"Alpha: {alpha}")
print(f"Beta: {beta}")

A high beta means “good TSLA picks” can still lose money on broad risk-off days, because the market component dominates short horizons. Printing alpha and beta makes that exposure visible instead of guessing from PnL.

## Build a beta-hedged return series

Construct a simple hedge by shorting beta units of SPY against one unit of TSLA, then re-estimate beta to confirm the market exposure was actually removed.

In [ ]:
portfolio = -1 * beta * benchmark_returns + asset_returns
portfolio.name = "TSLA + Hedge"
P = portfolio.values
alpha, beta = linreg(X, P)
print(f"Alpha: {alpha}")
print(f"Beta: {beta}")

This is the key workflow move: the hedged series is the part of returns that’s less explained by the market, which is closer to what we mean by “alpha.” In practice, desks often do this on a rolling window because beta changes through time, but the one-shot version teaches the core idea cleanly.

## Visualize exposure and hedge effect

Plot return series and the regression line so we can see both time-domain behavior (daily swings) and cross-sectional behavior (slope vs SPY) in one place.

In [ ]:
X2 = np.linspace(X.min(), X.max(), 100)
Y_hat = X2 * beta + alpha

In [ ]:
asset_returns.plot()
benchmark_returns.plot()
plt.ylabel("Daily Return")
plt.legend()
plt.show()

In [ ]:
plt.scatter(X, Y, alpha=0.3)
plt.xlabel("SPY Daily Return")
plt.ylabel("TSLA Daily Return")
plt.plot(X2, Y_hat, "r", alpha=0.9)
plt.show()

In [ ]:
portfolio.plot(alpha=0.9)
benchmark_returns.plot(alpha=0.5)
asset_returns.plot(alpha=0.5)
plt.ylabel("Daily Return")
plt.legend()
plt.show()

The scatter plot gives an intuitive beta picture: the tighter and steeper the cloud, the more “market-driven” the asset is over this period. Overlaying the hedged series against TSLA and SPY helps us verify the hedge is doing what we intended—reducing co-movement—so we can evaluate skill without being fooled by broad market moves.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.